## Chat Message Memory
- We need to store the historical chat messages in a efficient way
- It wraps another Runnable and manages the chat message history for it.
- Specifically, it loads previous messages in the conversation BEFORE passing it to the Runnable, and it saves the generated response as a message AFTER calling the runnable.
- This class also enables multiple conversations by saving each conversation with a session_id
- it then expects a session_id to be passed in the config when calling the runnable, and uses that to look up the relevant conversation history

In [1]:
! pip install langchain-ollama langchain-core

In [31]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate)
from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model_name = 'qwen3:0.6b'

llm = ChatOllama(
    base_url=base_url,
    model = model_name,
    # temperature=0.3,
    # num_predict=1024
)

template = ChatPromptTemplate.from_template("{prompt}")
chain = template | llm | StrOutputParser()

about = "My name is Neel Contractor. I am studing AI Agents"
chain.invoke({"prompt": about})

"Neel Contractor's AI Agent Study can be structured to cover both foundational concepts and practical applications, tailored to his academic focus. Here's a breakdown:\n\n### **1. Core Concepts**\n- **AI Agents**: Explain their role as autonomous systems (e.g., robots, autonomous vehicles). Mention key features like decision-making, learning, and adaptability.\n- **Machine Learning**: Discuss algorithms like supervised, unsupervised, and reinforcement learning. Highlight how these enable agents to learn from data.\n- **Neural Networks**: Simplify their function as models that mimic human brain structures for complex problem-solving.\n\n### **2. Applications**\n- **Robotics**: Examples include autonomous drones or industrial robots.\n- **Autonomous Systems**: Applications in healthcare (e.g., medical devices) or logistics (e.g., delivery robots).\n- **Healthcare**: AI agents could assist in diagnostics or patient monitoring.\n\n### **3. Ethical Considerations**\n- **Bias and Fairness**:

In [25]:
prompt = "What is my name?"
chain.invoke({"prompt":prompt})

"I don't have a name, but I'm here to help with your questions! Let me know how I can assist you. 😊"

## Runnable With Message History
In order to properly set this up there are two main things to consider:

- How to store and load messages?
- What is the underlying Runnable you are wrapping and what are its inputs/outputs?

In [20]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [29]:
def get_session_history(session_id):
    return SQLChatMessageHistory(session_id, connection="sqlite:///chat_history.db")

In [30]:
runnable_with_history = RunnableWithMessageHistory(chain, get_session_history)

In [32]:
user_id = "neel_contractor"
history = get_session_history(user_id)

history.get_messages()

[]

In [33]:
history.clear()

In [34]:
about

'My name is Neel Contractor. I am studing AI Agents'

In [35]:
runnable_with_history.invoke([HumanMessage(content=about)],
                            config={'configurable': {'session_id': user_id}})

'Hi! Nice to meet you! Your message is interesting — I’m a Contractor studying AI agents. Let me know if you need help with anything! 😊'

In [36]:
runnable_with_history.invoke([HumanMessage(content="whats my name?")],
                            config={'configurable':{'session_id':user_id}})

"Your name is Neel Contractor. I'm a Contractor studying AI agents. Let me know if you need help with anything! 😊"

## Message History with Dictionary Like Inputs

In [37]:
from langchain_core.prompts import MessagesPlaceholder, HumanMessagePromptTemplate, SystemMessagePromptTemplate

In [39]:
system = SystemMessagePromptTemplate.from_template("You are helpful assistant.")
human = HumanMessagePromptTemplate.from_template("{input}")

messages = [system, MessagesPlaceholder(variable_name='history'), human]

prompt = ChatPromptTemplate(messages=messages)

chain = prompt | llm | StrOutputParser()

runnable_with_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key='input', history_messages_key='history')

In [40]:
def chat_with_llm(session_id, input):
    output = runnable_with_history.invoke(
        {'input':input},
        config={'configurable':{'session_id':session_id}}
    )
    return output

In [42]:
user_id = "contractor"
chat_with_llm(user_id, about)

"Your interest in AI agents is fascinating! AI agents, or autonomous agents, are revolutionizing industries by enabling automation, decision-making, and data-driven processes. Here's how they're shaping the future:\n\n1. **Automation & Efficiency**: AI agents can perform tasks like inventory management, customer service, or logistics, reducing human effort while improving accuracy.  \n2. **Decision-Making**: They analyze vast amounts of data to make informed decisions, optimizing processes in healthcare, finance, and manufacturing.  \n3. **Innovation**: By bridging human and machine capabilities, AI agents drive innovation in fields like healthcare, energy, and education.  \n\nAI agents are not just tools—they’re transforming industries, making complex tasks easier and more efficient. As you study this field, I’d love to explore how AI agents impact real-world applications or how you can contribute to their development. Let me know how I can help! 😊"

In [43]:
chat_with_llm(user_id, 'what is my name?')

'Hello! My name is Neel Contractor. How can I assist you in your studies or explore AI agents further? 😊'